# 05 — Evaluación de Modelos

**TFM RunnAing** | Fase 5

**Objetivos:**
- Evaluar todos los modelos sobre la partición de **test** (15% usuarios)
- Métricas: MAE, RMSE, R²
- Tabla comparativa de todos los modelos
- Scatter plot predicho vs real (mejor modelo)
- Residual plot (mejor modelo)

**Outputs**: `reports/evaluation/metrics.csv` + `reports/figures/`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.splits import group_train_val_test_split, verify_no_user_overlap

np.random.seed(42)

MODELS_DIR = Path('../models')
EVAL_DIR = Path('../reports/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. Carga de datos y modelos

In [ ]:
# Cargar dataset procesado
df = pd.read_parquet('../data/processed/sessions_features.parquet')

# Cargar metadatos del split
with open(MODELS_DIR / 'split_meta.json') as f:
    split_meta = json.load(f)

FEATURE_COLS = split_meta['feature_cols']
best_model_name = split_meta['best_model_name']
print(f'Features: {FEATURE_COLS}')
print(f'Mejor modelo: {best_model_name}')

X = df[FEATURE_COLS]
y = df['trimp']
groups = df['userId']

# Recrear el mismo split (seed=42 garantiza reproducibilidad)
X_train, X_val, X_test, y_train, y_val, y_test, g_train, g_val, g_test = \
    group_train_val_test_split(X, y, groups, train_frac=0.70, val_frac=0.15, seed=42)

verify_no_user_overlap(g_train, g_val, g_test)
print(f'\nTest: {len(X_test):,} sesiones | {g_test.nunique():,} usuarios')

In [ ]:
# Cargar todos los modelos entrenados
MODEL_NAMES = ['LinearRegression', 'RandomForest', 'GradientBoosting', 'XGBoost', 'LightGBM']
models = {}
for name in MODEL_NAMES:
    path = MODELS_DIR / f'{name.lower()}_model.pkl'
    with open(path, 'rb') as f:
        models[name] = pickle.load(f)
    print(f'Cargado: {name}')

## 2. Evaluación sobre test — MAE, RMSE, R²

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    preds = model.predict(X_test)
    predictions[name] = preds

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)

    results.append({'modelo': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'{name:<22}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

metrics_df = pd.DataFrame(results).sort_values('MAE')
print()
print('=== TABLA COMPARATIVA (ordenada por MAE) ===')
print(metrics_df.to_string(index=False))

metrics_df.to_csv(EVAL_DIR / 'metrics.csv', index=False)
print('\nGuardado: reports/evaluation/metrics.csv')

## 3. Scatter plot: TRIMP predicho vs TRIMP real (mejor modelo)

In [ ]:
best_preds = predictions[best_model_name]
y_test_arr = y_test.values

best_mae  = mean_absolute_error(y_test_arr, best_preds)
best_r2   = r2_score(y_test_arr, best_preds)

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test_arr, best_preds, alpha=0.25, s=8, color='steelblue', label='Sesiones test')

lims = [min(y_test_arr.min(), best_preds.min()), max(y_test_arr.max(), best_preds.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')

ax.set_xlabel('TRIMP real', fontsize=12)
ax.set_ylabel('TRIMP predicho', fontsize=12)
ax.set_title(f'TRIMP predicho vs real — {best_model_name}\nMAE={best_mae:.2f} | R²={best_r2:.3f}',
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eval_scatter_predicho_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eval_scatter_predicho_vs_real.png')

## 4. Residual plot (mejor modelo)

In [ ]:
residuals = y_test_arr - best_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuos vs predichos
axes[0].scatter(best_preds, residuals, alpha=0.25, s=8, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('TRIMP predicho', fontsize=11)
axes[0].set_ylabel('Residuo (real - predicho)', fontsize=11)
axes[0].set_title('Residuos vs TRIMP predicho', fontsize=11)
axes[0].grid(alpha=0.3)

# Histograma de residuos
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals.mean(), color='orange', linestyle=':',
                linewidth=1.5, label=f'Media={residuals.mean():.2f}')
axes[1].set_xlabel('Residuo', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribución de residuos', fontsize=11)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(f'Análisis de residuos — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eval_residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eval_residual_plot.png')

## 5. Gráfico comparativo de MAE por modelo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#e74c3c' if m == best_model_name else 'steelblue' for m in metrics_df['modelo']]

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    bars = ax.barh(metrics_df['modelo'], metrics_df[metric],
                   color=colors, edgecolor='white')
    ax.set_title(f'{metric} en test', fontsize=12)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_width() * 0.98, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Comparativa de modelos — partición test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eval_comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/eval_comparativa_modelos.png')

In [ ]:
print('=== RESUMEN FASE 5 ===')
best_row = metrics_df[metrics_df['modelo'] == best_model_name].iloc[0]
print(f'Mejor modelo : {best_model_name}')
print(f'  MAE        : {best_row["MAE"]:.4f}')
print(f'  RMSE       : {best_row["RMSE"]:.4f}')
print(f'  R²         : {best_row["R2"]:.4f}')
print(f'\nOutputs:')
print(f'  reports/evaluation/metrics.csv')
print(f'  reports/figures/eval_scatter_predicho_vs_real.png')
print(f'  reports/figures/eval_residual_plot.png')
print(f'  reports/figures/eval_comparativa_modelos.png')
print(f'\nPróximo paso: notebook 06_shap_interpretability.ipynb')